# 01 — Ingest

**Project:** *marvel* — the Marvel Cinematic Universe as a part-of-whole box-office picture.

This notebook fetches the raw per-film box-office data and builds the small Phase/Saga
reference table, then loads everything into DuckDB with source provenance. Nothing is
transformed here — raw HTML lands in `data/raw/` untouched; cleaning happens in `02-clean`.

**Sources (see `SOURCES.md` for full attribution):**
- **The Numbers — MCU franchise page** (`tnumbers_mcu`, PRIMARY). One HTML table carrying,
  per film: release date, production budget, opening weekend, **domestic** and **worldwide**
  lifetime gross (nominal $). This is the only single-page source with worldwide-per-film,
  which the treemap needs for tile sizing.
- **Box Office Mojo — MCU franchise page** (`bom_mcu`, CROSS-CHECK). Domestic lifetime gross
  per film + distributor + release date. Used in `02-clean` to sanity-check The Numbers'
  domestic figures and confirm the roster.
- **MCU Phase / Saga groupings** — a small factual lookup (title → phase → saga) built inline
  below from Marvel Studios' official Phase designations (verified against Wikipedia). Not
  scraped; it's reference data.

**Scope decision (theatrical MCU only):** released theatrical feature films in Marvel
Studios' Phase 1–6 canon, including the Sony-distributed Tom Holland *Spider-Man* films
(MCU-canon). Excluded: unreleased/future films, TV specials (e.g. *Werewolf by Night*), and
Disney+ series. The filtering itself happens in `02-clean`; here we just capture the raw tables.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config, ingest_source
from src.clean_quality import get_connection, load_to_duckdb, register_source, run_sql

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Source 1 — The Numbers MCU franchise page (PRIMARY)

Fetches `https://www.the-numbers.com/movies/franchise/Marvel-Cinematic-Universe`. The page's
first `<table>` is the full franchise box-office history. `ingest_source()` saves the raw HTML
to `data/raw/tnumbers_mcu.html` and returns the parsed table.

The raw table includes announced-but-unreleased films (shown with `$0` grosses) and one TV
special — we keep them in the raw table as-is and filter them out in `02-clean` so the raw
capture stays faithful to the source.

In [ ]:
df_tn = ingest_source('tnumbers_mcu', cfg, rate_limit_seconds=1.5)
print('raw shape:', df_tn.shape)
print('columns:', list(df_tn.columns))
df_tn.head(10)

Load the raw table into DuckDB as `tnumbers_mcu_raw` (verbatim — no cleaning) and register
its provenance in the `_sources` metadata table.

In [ ]:
load_to_duckdb(df_tn, 'tnumbers_mcu_raw', con)
register_source(
    con,
    table='tnumbers_mcu_raw',
    name='The Numbers — Marvel Cinematic Universe franchise box office',
    url='https://www.the-numbers.com/movies/franchise/Marvel-Cinematic-Universe',
    license='Proprietary (industry aggregator); cited reference for a fun-tier project',
    notes=('Per-film release date, production budget, opening weekend, domestic and worldwide '
           'lifetime gross. Primary source (only single page with worldwide-per-film). '
           'Unreleased films appear with $0 and are filtered in 02-clean; Werewolf by Night '
           '(TV special) also dropped. Theatrical released-only universe.'),
    retrieved='2026-09-19',
    methodology=('Aggregates studio-reported/tracked theatrical receipts. Domestic = U.S. & '
                 'Canada; worldwide = domestic + international. Budgets = production cost.'),
    series_breaks=('NOMINAL year-of-release dollars across all years — not inflation-adjusted, '
                   'so raw cross-era gross comparisons overstate later films. Recent films\' '
                   'worldwide totals may still be settling; FX-sensitive.'),
)
print('registered tnumbers_mcu_raw')

## Source 2 — Box Office Mojo MCU franchise page (CROSS-CHECK)

Fetches `https://www.boxofficemojo.com/franchise/fr541495045/`. BOM's franchise table is
**domestic only** (no worldwide per film), so it isn't the primary source — but it's a good
independent check on The Numbers' domestic figures and confirms distributors and the roster.
The raw table also contains re-release rows and a "Columbia 100th Anniversary Series" entry,
which we leave in the raw capture and filter in `02-clean`.

In [ ]:
df_bom = ingest_source('bom_mcu', cfg, rate_limit_seconds=1.5)
print('raw shape:', df_bom.shape)
print('columns:', list(df_bom.columns))
df_bom.head(10)

In [ ]:
load_to_duckdb(df_bom, 'bom_mcu_raw', con)
register_source(
    con,
    table='bom_mcu_raw',
    name='Box Office Mojo — Marvel Cinematic Universe franchise page',
    url='https://www.boxofficemojo.com/franchise/fr541495045/',
    license='Proprietary (IMDbPro); cited reference for a fun-tier project',
    notes=('Domestic (U.S. & Canada) lifetime gross per film + max theaters, opening, release '
           'date, distributor. Cross-check on The Numbers domestic figures and roster. '
           'Re-release + Columbia-anniversary rows filtered in 02-clean.'),
    retrieved='2026-09-19',
    methodology='Tracked/reported U.S. & Canada theatrical receipts. Domestic only; nominal $.',
    series_breaks='Nominal dollars; re-release runs listed as separate rows (excluded in cleaning).',
)
print('registered bom_mcu_raw')

## Source 3 — MCU Phase / Saga reference table (factual lookup)

The Phase and Saga each film belongs to is **not** in the box-office data — it's a factual
grouping from Marvel Studios' official Phase announcements. We build it inline as a small
lookup (`film` → `phase` → `saga`), verified against the Wikipedia *List of Marvel Cinematic
Universe films* and per-Phase articles (fun-tier; crowd-edited reference is acceptable and is
cited plainly in `SOURCES.md`).

**Boundaries used (authoritative as of 2026-09):**
- **Infinity Saga** — Phase 1 (2008–2012), Phase 2 (2013–2015), Phase 3 (2016–2019)
- **Multiverse Saga** — Phase 4 (2021–2022), Phase 5 (2023–2025), Phase 6 (2025– )

Titles below are written to match The Numbers' spellings so they join cleanly in `02-clean`
(any residual mismatches are reconciled there). This is the authoritative roster of the
**released** theatrical MCU films.

In [ ]:
# film title (matching The Numbers) -> (phase, saga)
PHASE_MAP = [
    # ── Infinity Saga · Phase 1 (2008-2012) ──
    ('Iron Man', 1),
    ('The Incredible Hulk', 1),
    ('Iron Man 2', 1),
    ('Thor', 1),
    ('Captain America: The First Avenger', 1),
    ('The Avengers', 1),
    # ── Infinity Saga · Phase 2 (2013-2015) ──
    ('Iron Man 3', 2),
    ('Thor: The Dark World', 2),
    ('Captain America: The Winter Soldier', 2),
    ('Guardians of the Galaxy', 2),
    ('Avengers: Age of Ultron', 2),
    ('Ant-Man', 2),
    # ── Infinity Saga · Phase 3 (2016-2019) ──
    ('Captain America: Civil War', 3),
    ('Doctor Strange', 3),
    ('Guardians of the Galaxy Vol 2', 3),
    ('Spider-Man: Homecoming', 3),
    ('Thor: Ragnarok', 3),
    ('Black Panther', 3),
    ('Avengers: Infinity War', 3),
    ('Ant-Man and the Wasp', 3),
    ('Captain Marvel', 3),
    ('Avengers: Endgame', 3),
    ('Spider-Man: Far From Home', 3),
    # ── Multiverse Saga · Phase 4 (2021-2022) ──
    ('Black Widow', 4),
    ('Shang-Chi and the Legend of the Ten Rings', 4),
    ('Eternals', 4),
    ('Spider-Man: No Way Home', 4),
    ('Doctor Strange in the Multiverse of Madness', 4),
    ('Thor: Love and Thunder', 4),
    ('Black Panther: Wakanda Forever', 4),
    # ── Multiverse Saga · Phase 5 (2023-2025) ──
    ('Ant-Man and the Wasp: Quantumania', 5),
    ('Guardians of the Galaxy Vol 3', 5),
    ('The Marvels', 5),
    ('Deadpool & Wolverine', 5),
    ('Captain America: Brave New World', 5),
    ('Thunderbolts*', 5),
    # ── Multiverse Saga · Phase 6 (2025- ) ──
    ('The Fantastic Four: First Steps', 6),
    ('Spider-Man: Brand New Day', 6),
]

def saga_for(phase: int) -> str:
    return 'Infinity Saga' if phase <= 3 else 'Multiverse Saga'

df_phase = pd.DataFrame(
    [(film, phase, f'Phase {phase}', saga_for(phase)) for film, phase in PHASE_MAP],
    columns=['film', 'phase_num', 'phase', 'saga'],
)
print('phase rows:', len(df_phase))
df_phase.groupby(['saga', 'phase'], sort=False).size().rename('films')

In [ ]:
load_to_duckdb(df_phase, 'phase_map', con)
register_source(
    con,
    table='phase_map',
    name='MCU Phase / Saga groupings (hand-built factual lookup)',
    url='https://en.wikipedia.org/wiki/List_of_Marvel_Cinematic_Universe_films',
    license='Factual groupings (not copyrightable); verified against Wikipedia (CC-BY-SA)',
    notes=('title -> phase (1-6) -> saga (Infinity=P1-3, Multiverse=P4-6). Authoritative '
           'roster of released theatrical MCU films; Sony Tom Holland Spider-Man films '
           'included as MCU-canon.'),
    retrieved='2026-09-19',
    methodology='Marvel Studios official Phase announcements, cross-checked against Wikipedia.',
    series_breaks='N/A (factual reference, not a measured time series).',
)
print('registered phase_map')

## Provenance check

All three raw tables are now in DuckDB with `_sources` entries.

In [ ]:
run_sql('SELECT duckdb_table, source_name, retrieved FROM _sources ORDER BY duckdb_table', con)

---
**Next:** `02-clean.ipynb` — parse the dollar/date strings, filter to released theatrical
films, join the Phase/Saga map, cross-check domestic against BOM, and save interim Parquet.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')